# 第2章 — フィッシングサイトと迷惑メールの検出

本書第2章（pp.13–39）では、フィッシングサイトと迷惑メールの分類を通して、
読み込み、分割、ハイパーパラメータ探索、学習、評価を一巡する。
サイトの分類には既存の数値特徴量を、メールの分類には本文から作るTF-IDFを使う。
同じ二値分類でも、入力をどう用意するかが異なる。

| 本書 | 入力と手法 | 実習中に確認すること |
|---|---|---|
| §2.1、pp.13–14 | ソーシャルエンジニアリング | 分類器が観測できる情報の範囲 |
| §2.2.1、pp.14–26 | 30特徴量＋ロジスティック回帰 | スコア、正則化、評価指標 |
| §2.2.2、pp.26–29 | 同じ30特徴量＋決定木 | 条件分岐と過学習 |
| §2.3.1–2.3.2、pp.29–38 | 英語メール＋TF-IDF＋LightGBM | テキストの数値化とデータの偏り |
| §2.5、p.39 | Ling-Spamを使った演習 | 別のコーパスで検証する手順 |

節・ページは本書に対応する。ロジスティック回帰、決定木、TF-IDF、勾配ブースティングの数式は、
コードを説明するための**補足**として加えた。図はこの実習用の作図であり、本書の画面や図の転載ではない。

## §2.1 ソーシャルエンジニアリングと分類器の入力

本書は、なりすましや報酬による誘導など、人を介して機密情報を得る手口を挙げている。
フィッシングでは、組織やサービスを装った連絡から偽サイトへ誘導し、認証情報などを入力させる。
ただし、この実習の分類器は、被害者と攻撃者のやり取り全体を読んでいるわけではない。
サイト側は30個の特徴量、メール側は本文の単語表現だけを観測する。

したがって、出力が陰性でも「詐欺ではない」とまでは言えない。
観測した入力から、学習データ上の正常例に近いと判断した、という範囲の結果になる。
メールのspamラベルもフィッシング専用のラベルではない。この2つの課題を混同しないようにしたい。

新しいCPUランタイムで **ランタイム → すべてのセルを実行** を選ぶ。
第1章の事前実行、GPU、Driveのマウント、GitHub認証は不要。実サイトへの接続やメール送信は行わない。

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *[
            "numpy==2.2.6",
            "pandas==2.2.3",
            "scipy==1.15.3",
            "scikit-learn==1.7.2",
            "matplotlib==3.10.3",
            "optuna==4.5.0",
            "lightgbm==4.6.0",
        ],
    ],
    check=True,
)

In [ ]:
import platform
from importlib.metadata import version

import matplotlib.pyplot as plt

print(f"Python {platform.python_version()} | CPU runtime")
for package in ("numpy", "pandas", "scipy", "scikit-learn", "optuna", "lightgbm"):
    print(f"{package}: {version(package)}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. データ取得と監査 — §2.2.1、pp.15–19／§2.3.2、pp.31–33

### サイトを30個の特徴量で表す

本書のフィッシングデータは、UCIのPhishing Websitesである。
元のARFFには列名や値の候補があり、本書では扱いやすいCSVへ変換している。
1行は1件の観測、最初の30列が特徴量、最後の1列が正解ラベルになる。

$$
X\in\{-1,0,1\}^{n\times30},\qquad y\in\{0,1\}^{n}
$$

CSVの数値は、元のURLやHTMLそのものではなく、抽出済みの性質を符号化したものだ。
たとえば次のような情報が含まれる。

| 特徴量 | 表している情報の例 |
|---|---|
| `having_IP_Address` | URLのホストにIPアドレスを使うか |
| `URL_Length` | URLの長さの区分 |
| `having_Sub_Domain` | サブドメインの構成 |
| `SSLfinal_State` | HTTPS・証明書についての評価 |
| `URL_of_Anchor` | アンカーのリンク先についての評価 |
| `age_of_domain` | ドメインの経過期間の区分 |

符号の意味は特徴量ごとの定義で確認する。`1`を一律に「特徴が存在する」、`0`を「正常」と読んではいけない。
このノートブックでは抽出済みCSVを使うため、任意のURLから同じ30列を作る機能は実装していない。

**正解ラベルは、元データの`-1`がフィッシング、`+1`が正常**である。
`load_phishing`では`Result == -1`を`1`へ変換し、危険側を陽性に揃える。
この変換は特徴量の符号を反転する操作ではない。ラベルの列だけに行う。

### メールは本文とspam/hamの組で読む

Enron-Spamは、Enron由来の正当なメールと迷惑メールをラベル付きで整理したコーパスである。
ここで使う`enron1`は、hamが3,672通、spamが1,500通。
`ham`フォルダを0、`spam`フォルダを1とし、本文を読み込む。
ファイル名や格納フォルダを予測の特徴量にすると、ラベルの付け方を読めてしまうため入力には渡さない。

### 分割の前に重複を調べる理由

同じメールが訓練とテストの両方に入ると、未知のメールへの性能を測りにくくなる。
サイトの表でも、同じ特徴量ベクトルが分割をまたぐと、見かけの汎化性能が高く出ることがある。
そこで、空の入力を除き、同じ入力に矛盾するラベルが付いた行を全件除外し、残りの同一入力を1件にまとめる。
メールの同一性は、小文字化して空白を揃えた本文で判定する。学習に渡す本文は元のまま保持する。

| データ | 元の件数 | 矛盾ラベルで除外 | 重複で除外 | 残る件数 |
|---|---:|---:|---:|---:|
| Phishing Websites | 11,055 | 357 | 4,977 | 5,721 |
| Enron1 | 5,172 | 0 | 277 | 4,895 |

この集計は固定した配布ファイルに対する結果で、次のセルでも出力する。
同じ30特徴量を持つ別サイトは区別できないため、サイト側では「重複サイトを特定した」とまでは言えない。
近似した本文、同一キャンペーン、同じ送信者のメールも、これだけでは取り除けない。

配布元のコミットとSHA-256を固定し、取得したバイト列が想定したものかを照合する。
ZIPは展開せずに読み、パスとサイズを検査する。これらは取得手順の検査であり、ラベルの正しさを保証する処理ではない。

In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path, PurePosixPath
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd

UPSTREAM_REVISION = "565def0834d063281ff0776918ad829020919752"
DATA_URL = (
    f"https://raw.githubusercontent.com/oreilly-japan/ml-security-jp/{UPSTREAM_REVISION}/ch02"
)
DATA_HASHES = {
    "dataset.csv": "98395653cdd0e5a79c73bc59907e63ce82681d40947858b73464c08981a3629c",
    "enron1.zip": "97d1e3942057e7a9192af73adc70dad1f4279a05dadde2c1606a364c245519ed",
}
FEATURE_NAMES = (
    "having_IP_Address",
    "URL_Length",
    "Shortining_Service",
    "having_At_Symbol",
    "double_slash_redirecting",
    "Prefix_Suffix",
    "having_Sub_Domain",
    "SSLfinal_State",
    "Domain_registeration_length",
    "Favicon",
    "port",
    "HTTPS_token",
    "Request_URL",
    "URL_of_Anchor",
    "Links_in_tags",
    "SFH",
    "Submitting_to_email",
    "Abnormal_URL",
    "Redirect",
    "on_mouseover",
    "RightClick",
    "popUpWidnow",
    "Iframe",
    "age_of_domain",
    "DNSRecord",
    "web_traffic",
    "Page_Rank",
    "Google_Index",
    "Links_pointing_to_page",
    "Statistical_report",
)

In [ ]:
def fetch_dataset(name: str, cache_dir: Path) -> Path:
    expected = DATA_HASHES[name]
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    target = cache_dir / name
    if target.exists():
        if sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Checksum mismatch: {target}. Remove the cached file and retry.")
        return target

    temporary = None
    try:
        with urlopen(f"{DATA_URL}/{name}", timeout=60) as response:
            with NamedTemporaryFile(dir=cache_dir, delete=False) as stream:
                temporary = Path(stream.name)
                digest = sha256()
                size = 0
                while chunk := response.read(1024 * 1024):
                    size += len(chunk)
                    if size > 32 * 1024 * 1024:
                        raise ValueError("Dataset exceeds the download limit")
                    digest.update(chunk)
                    stream.write(chunk)
        if digest.hexdigest() != expected:
            raise ValueError(f"Checksum mismatch: {name}")
        temporary.replace(target)
        return target
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

In [ ]:
@dataclass(frozen=True)
class DataAudit:
    raw_rows: int
    empty_rows: int
    conflicting_rows: int
    duplicate_rows: int
    retained_rows: int

In [ ]:
def deduplicate(frame: pd.DataFrame, keys: list[str]) -> tuple[pd.DataFrame, DataAudit]:
    frame = frame.copy()
    raw_rows = len(frame)
    empty = frame[keys].isna().any(axis=1)
    if keys == ["text"]:
        frame["_identity"] = frame["text"].fillna("").str.lower().str.split().str.join(" ")
        empty |= frame["_identity"].eq("")
        identity = ["_identity"]
    else:
        identity = keys
    frame = frame.loc[~empty].copy()
    conflicting = frame.groupby(identity, dropna=False)["label"].transform("nunique").gt(1)
    clean = frame.loc[~conflicting].drop_duplicates(identity)
    audit = DataAudit(
        raw_rows,
        int(empty.sum()),
        int(conflicting.sum()),
        len(frame) - int(conflicting.sum()) - len(clean),
        len(clean),
    )
    return clean.drop(columns="_identity", errors="ignore").reset_index(drop=True), audit

In [ ]:
def load_phishing(path: Path) -> tuple[pd.DataFrame, DataAudit]:
    values = pd.read_csv(path, header=None)
    if values.shape[1] != 31 or values.empty:
        raise ValueError("Expected 30 phishing features and one label")
    if values.isna().any().any() or not values.isin([-1, 0, 1]).all().all():
        raise ValueError("Invalid phishing feature values")
    if set(values.iloc[:, -1].unique()) != {-1, 1}:
        raise ValueError("Expected UCI labels -1 (phishing) and +1 (legitimate)")
    values.columns = [*FEATURE_NAMES, "label"]
    values = values.astype(np.int8)
    values["label"] = values["label"].eq(-1).astype(np.int8)
    return deduplicate(values, list(FEATURE_NAMES))

In [ ]:
def load_enron(path: Path) -> tuple[pd.DataFrame, DataAudit]:
    rows = []
    with ZipFile(path) as archive:
        members = archive.infolist()
        if len(members) > 10_000 or sum(m.file_size for m in members) > 64 * 1024 * 1024:
            raise ValueError("Enron archive exceeds the size limit")
        for member in sorted(members, key=lambda item: item.filename):
            if member.is_dir():
                continue
            if member.filename == "enron1/Summary.txt":
                continue
            parts = PurePosixPath(member.filename).parts
            if len(parts) != 3 or parts[0] != "enron1" or parts[1] not in {"ham", "spam"}:
                raise ValueError(f"Unexpected archive member: {member.filename}")
            if member.file_size > 1024 * 1024:
                raise ValueError(f"Oversized email: {member.filename}")
            text = archive.read(member).decode("utf-8", errors="replace")
            rows.append({"text": text, "label": int(parts[1] == "spam")})
    if not rows or {row["label"] for row in rows} != {0, 1}:
        raise ValueError("Enron archive must contain both ham and spam")
    return deduplicate(pd.DataFrame(rows), ["text"])

In [ ]:
cache_dir = Path("data")
phishing, phishing_audit = load_phishing(fetch_dataset("dataset.csv", cache_dir))
emails, email_audit = load_enron(fetch_dataset("enron1.zip", cache_dir))
display(pd.DataFrame({"phishing": vars(phishing_audit), "enron1": vars(email_audit)}).T)
display(
    pd.DataFrame(
        {
            "phishing": phishing.label.value_counts(),
            "enron1": emails.label.value_counts(),
        }
    ).rename(index={0: "legitimate / ham", 1: "phishing / spam"})
)

## 2. 訓練・検証・テストを分ける — §2.2.1、pp.19–23

本書では20%をテストに回し、残りの訓練データで交差検証する。
この実習では、判定閾値を調整するデータも独立に確保し、訓練60%、検証20%、最終テスト20%に分ける。
3つの部分を使う目的はそれぞれ異なる。

![訓練60%、検証20%、最終テスト20%と、訓練内部の3分割交差検証](https://raw.githubusercontent.com/black-leg-nameko/security-ml-sakuraiLab/main/notebooks/figures/02_split_cv.png)

*図2　CVで使う評価foldも、図の60%の中にある。右端の最終テスト20%は探索にも閾値選択にも使わない。*

| データ | 何を決めるか | コードとの対応 |
|---|---|---|
| 訓練60% | 分類器の重み、木の分岐、標準化、語彙・IDF | `split.train`、`tune_model` |
| 訓練内のCV評価fold | ハイパーパラメータの比較 | `StratifiedKFold`、`cross_val_score` |
| 検証20% | 誤検知率を見ながら閾値を選ぶ | `split.validation`、`select_threshold` |
| 最終テスト20% | 決めた構成の性能を報告する | `split.test`、`binary_metrics` |

層化は、各部分でクラスの比率が大きく崩れないようにする処理である。
乱数の固定は同じ分割を作り直すために行う。これによって時系列やキャンペーンの独立性まで成立するわけではない。

### 交差検証とOptuna

訓練部分を$K$個のfoldに分け、1つを評価用、残りを学習用として$K$回繰り返す。
ハイパーパラメータ$h$の良さを、ここでは各foldのF1の平均で比べる。

$$
\operatorname{CV}(h)=\frac{1}{K}\sum_{k=1}^{K}
\operatorname{F1}\!\left(y^{(k)},\widehat y_h^{(k)}\right)
$$

Optunaは試した設定とスコアの履歴を使い、次に試す設定を提案する。
本書は正解率を指標に時間上限を設けるが、この実装は`folds=3`、各モデル`trials=8`、指標をF1にしている。
CVが最良の設定も、最終テストや別の時期のデータで必ず最良になるとは限らない。

前処理も各foldで作り直す必要がある。全件で標準化やTF-IDFを学習してからCVすると、
評価foldの情報が訓練側へ入ってしまう。このため前処理と分類器を`Pipeline`に収め、Pipeline全体をCVに渡す。

In [ ]:
from typing import Any

import optuna
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 101
    trials: int = 8
    folds: int = 3
    max_false_positive_rate: float = 0.01

    def __post_init__(self):
        if self.trials < 1 or self.folds < 2:
            raise ValueError("trials >= 1 and folds >= 2 are required")
        if not 0 <= self.max_false_positive_rate <= 1:
            raise ValueError("max_false_positive_rate must lie in [0, 1]")

In [ ]:
@dataclass(frozen=True)
class Split:
    train: np.ndarray
    validation: np.ndarray
    test: np.ndarray

In [ ]:
def split_indices(y, seed: int) -> Split:
    indices = np.arange(len(y))
    train_validation, test = train_test_split(
        indices,
        test_size=0.2,
        random_state=seed,
        stratify=y,
    )
    train, validation = train_test_split(
        train_validation,
        test_size=0.25,
        random_state=seed,
        stratify=np.asarray(y)[train_validation],
    )
    return Split(train, validation, test)

In [ ]:
def make_model(kind: str, params: dict[str, Any], seed: int) -> Pipeline:
    if kind == "logistic":
        return Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        C=params["C"],
                        solver="lbfgs",
                        max_iter=2000,
                        random_state=seed,
                    ),
                ),
            ]
        )
    if kind == "tree":
        return Pipeline([("classifier", DecisionTreeClassifier(**params, random_state=seed))])
    if kind == "spam":
        return Pipeline(
            [
                (
                    "tfidf",
                    TfidfVectorizer(
                        stop_words="english",
                        min_df=2,
                        max_features=20_000,
                        sublinear_tf=True,
                        dtype=np.float32,
                    ),
                ),
                (
                    "classifier",
                    LGBMClassifier(
                        **params,
                        objective="binary",
                        verbosity=-1,
                        n_jobs=2,
                        random_state=seed,
                        deterministic=True,
                        force_col_wise=True,
                        importance_type="gain",
                    ),
                ),
            ]
        )
    raise ValueError(f"Unknown model: {kind}")

In [ ]:
def suggest_params(trial: optuna.Trial, kind: str) -> dict[str, Any]:
    if kind == "logistic":
        return {"C": trial.suggest_float("C", 1e-3, 100, log=True)}
    if kind == "tree":
        return {
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_depth": trial.suggest_int("max_depth", 3, 24),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 16),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 1e-2, log=True),
        }
    if kind == "spam":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 60, 160, step=20),
            "num_leaves": trial.suggest_int("num_leaves", 7, 31),
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 40),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        }
    raise ValueError(f"Unknown model: {kind}")

In [ ]:
def tune_model(kind: str, X_train, y_train, config: ExperimentConfig):
    cv = StratifiedKFold(n_splits=config.folds, shuffle=True, random_state=config.seed)

    def objective(trial):
        model = make_model(kind, suggest_params(trial, kind), config.seed)
        with threadpool_limits(limits=2):
            scores = cross_val_score(
                model,
                X_train,
                y_train,
                cv=cv,
                scoring="f1",
                n_jobs=1,
                error_score="raise",
            )
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=config.seed),
    )
    study.optimize(objective, n_trials=config.trials, n_jobs=1)
    model = make_model(kind, study.best_params, config.seed)
    with threadpool_limits(limits=2):
        model.fit(X_train, y_train)
    return model, study

In [ ]:
config = ExperimentConfig(seed=101, trials=8, folds=3, max_false_positive_rate=0.01)
optuna.logging.set_verbosity(optuna.logging.WARNING)
phishing_split = split_indices(phishing.label, config.seed)
email_split = split_indices(emails.label, config.seed)
X_phishing = phishing.loc[:, list(FEATURE_NAMES)]
y_phishing = phishing.label
X_email = emails.text
y_email = emails.label
display(
    pd.DataFrame(
        {
            "phishing": {k: len(v) for k, v in vars(phishing_split).items()},
            "enron1": {k: len(v) for k, v in vars(email_split).items()},
        }
    ).T
)

## 3. ロジスティック回帰と決定木 — §2.2.1–2.2.2、pp.19–29

### 補足：ロジスティック回帰のスコア

名前に「回帰」と付くが、ここでは0/1の分類に使う。
特徴量の重み付き和を作り、シグモイド関数で0から1の範囲のスコアへ変換する。

$$
z=w^\mathsf{T}\widetilde{x}+b,\qquad
s(x)=\sigma(z)=\frac{1}{1+\exp(-z)}
$$

$\widetilde{x}$は標準化後の入力、$w$は係数、$b$は切片である。
スコアが0.5以上かどうかで分けると、境界は$w^\mathsf{T}\widetilde{x}+b=0$になる。
元の特徴量の空間で、線形な境界を作るモデルと考えられる。

![重み付き和zをスコアへ変換するシグモイドと、閾値0.5・0.8の比較](https://raw.githubusercontent.com/black-leg-nameko/security-ml-sakuraiLab/main/notebooks/figures/03_sigmoid.png)

*図3　閾値を0.5から0.8へ上げると、陽性とする範囲が狭くなる。これはモデルを学習し直す操作ではない。*

学習では、正解とスコアの食い違いを小さくしつつ、係数が極端になりすぎないようにする。
その考え方は、二値交差エントロピーと正則化を合わせた次の目的関数で表せる。
以下は説明用の表記であり、ライブラリ内部の定数倍まで指定するものではない。

$$
\mathcal{L}(w,b)=
-\frac{1}{n}\sum_{i=1}^{n}
\left[y_i\log s_i+(1-y_i)\log(1-s_i)\right]
+\frac{\lambda}{2}\lVert w\rVert_2^2
$$

`LogisticRegression`の`C`は正則化の強さの逆数に対応する。
小さい`C`ほど係数を強く抑える。今回の探索対象は`C`だけで、`max_iter=2000`は収束計算の上限として固定している。
最大反復回数を増やすこと自体が、モデルの表現力を増やすわけではない。

### 補足：決定木は条件で領域を分ける

決定木は、「この特徴量がこの値以下か」という分岐を重ねる。
ロジスティック回帰では表現しにくい、特徴量の組み合わせによる条件を作れる。

![特徴量の条件分岐を2段重ねた説明用の決定木](https://raw.githubusercontent.com/black-leg-nameko/security-ml-sakuraiLab/main/notebooks/figures/04_tree.png)

*図4　分岐の読み方を示す仮の木。学習した木や、UCI特徴量の正式な判定規則を再現した図ではない。*

たとえばGini不純度を使う場合、ノードに含まれるクラス比率を$p_c$とすると、不純度は次で与えられる。

$$
G(t)=1-\sum_{c\in\{0,1\}}p_c^2,\qquad
\Delta G=G(t)-\frac{n_L}{n_t}G(L)-\frac{n_R}{n_t}G(R)
$$

片方のクラスだけなら$G=0$、半々なら$G=0.5$である。
分岐後の重み付き不純度が下がる条件を選んでいく。
ただし細かく分け続けると、少数の訓練例にしか当てはまらない枝ができる。
`max_depth`で深さを、`min_samples_leaf`で葉に残す件数を制限するのは、この過学習を抑えるためだ。

| モデル | 探索する主な設定 | 読み方 |
|---|---|---|
| ロジスティック回帰 | `C` | 係数への正則化をどの程度緩めるか |
| 決定木 | `criterion` | Gini不純度かエントロピーか |
| 決定木 | `max_depth` / `min_samples_leaf` | 枝をどこまで細かくするか |
| 決定木 | `ccp_alpha` | 複雑な部分木をどの程度剪定するか |

本書の結果では決定木で改善が見られるが、常に決定木が優れるという法則ではない。
次のセルでは同じ分割で2つを比較し、**訓練内CVのF1**で採用するモデルを決める。
テストの正解率を見て採用モデルを決め直さない。

In [ ]:
models, studies = {}, {}
for kind in ("logistic", "tree"):
    models[kind], studies[kind] = tune_model(
        kind,
        X_phishing.iloc[phishing_split.train],
        y_phishing.iloc[phishing_split.train],
        config,
    )
    print(f"{kind}: CV F1 = {studies[kind].best_value:.4f}, params = {studies[kind].best_params}")
selected_phishing = max(("logistic", "tree"), key=lambda name: studies[name].best_value)
print(f"CV-selected phishing model: {selected_phishing}")

## 4. 英語メールをTF-IDFへ変換する — §2.3、pp.29–36

### 本書のNLP概論と、この実装の範囲

本書の図2-8（p.31）は、字句、構文、意味、談話、語用論という分析の段階を紹介している。
ただし、ここで作るTF-IDF分類器がそのすべてを実行するわけではない。
単語の出現から分類するため、否定、引用、文脈、送り手の意図を構文・意味解析として明示的に扱わない。
この表現では、同じ単語を含み、出現回数も同じ2つの文章は、語順が変わっても同じベクトルになる。

### 補足：TF、IDF、正規化

文書$d$に単語$t$が出る回数を$c_{t,d}$とする。
今回は`sublinear_tf=True`なので、回数の影響を対数で抑える。
IDFにはscikit-learnの既定の平滑化を使う。$N$は学習文書数、$\operatorname{df}(t)$は単語$t$を含む学習文書数である。

$$
\operatorname{tf}(t,d)=
\begin{cases}
1+\log c_{t,d} & c_{t,d}>0\\
0 & c_{t,d}=0
\end{cases}
\qquad
\operatorname{idf}(t)=1+\log\frac{1+N}{1+\operatorname{df}(t)}
$$

$$
v_{t,d}=\operatorname{tf}(t,d)\operatorname{idf}(t),\qquad
x_d=\frac{v_d}{\lVert v_d\rVert_2}
$$

同じ文書内で何度も出ればTFが上がり、学習文書の多くに出る単語は相対的にIDFが小さくなる。
IDFが小さいことと、その単語が分類に不要であることは同じではない。
L2正規化は文書ベクトルの長さを揃える処理で、単語の意味を理解する処理ではない。
語彙に残った単語が1つもない文書はゼロベクトルになる。

![3つの仮想メールに対する単語出現回数と、平滑化したTF-IDFの比較](https://raw.githubusercontent.com/black-leg-nameko/security-ml-sakuraiLab/main/notebooks/figures/05_tfidf.png)

*図5　説明用の3文書。`meeting`のdfは2、`prize`のdfは1なので、IDFはそれぞれ約1.29と1.69になる。
右図はL2正規化後の値。小例だけは`min_df=1`とし、実習本体の`min_df=2`とは変えている。*

3通目は語彙に残る単語が`prize`だけなので、正規化後はその成分が1になる。
単語を2回書けば最終的な特徴量も2倍になる、という計算ではない。

実装の語彙設定にも目を通しておく。

| 設定 | この実習での値 | 何が変わるか |
|---|---|---|
| `stop_words` | `english` | 一般的な英単語のリストを除外する |
| `min_df` | `2` | 訓練文書1件だけに出る単語を語彙から除く |
| `max_features` | `20000` | 語彙の上限を設ける。超えた場合は出現頻度に基づき選ぶ |
| `sublinear_tf` | `True` | 同一文書での反復の影響を対数で抑える |
| `dtype` | `float32` | TF-IDFの数値を32ビット浮動小数点で保持する |

`TfidfVectorizer`の既定のトークン抽出は、単に空白だけで文字列を切る処理とは異なる。
日本語メールで同じ設定を使えば同等の性能が出る、という前提も置けない。
本書の英語コーパスに合わせた実習である。

語彙が20,000語あっても、1通のメールにすべてが出るわけではない。
全体の多くは0になるため、非ゼロの要素を保持する疎行列のまま学習する。
例として5,000行×20,000列を密な`float32`配列にすると、数値部分だけで約400 MBになる。
`toarray()`で密にする必要はない。

### 補足：LightGBMは木を順に足す

本書§2.3.2（pp.33–36）では、勾配ブースティング木のLightGBMを使っている。
1本の木を深くする代わりに、その時点の損失を減らす方向へ次の木を追加していく。
二値分類の加法モデルは、概念的には次のように書ける。

$$
F_M(x)=F_0(x)+\eta\sum_{m=1}^{M}h_m(x),\qquad s(x)=\sigma(F_M(x))
$$

$h_m$が各段階の木、$\eta$が`learning_rate`、$M$が木の本数に対応する。
各木を独立に作って多数決するランダムフォレストとは学習の仕方が異なる。
`num_leaves`や`min_child_samples`は木の細かさを、`reg_lambda`は正則化を調整する。

本書は`LightGBMTunerCV`を使うが、ここではTF-IDFも各fold内で学習するため、
`TfidfVectorizer`と`LGBMClassifier`を同じPipelineに入れ、通常のOptunaで全体を評価する。
訓練全件から作った語彙・IDFを、CVの評価foldへ先回りして渡さないための構成である。

In [ ]:
models["spam"], studies["spam"] = tune_model(
    "spam",
    X_email.iloc[email_split.train],
    y_email.iloc[email_split.train],
    config,
)
print(f"spam: CV F1 = {studies['spam'].best_value:.4f}, params = {studies['spam'].best_params}")

## 5. 検証データで判定閾値を決める — 評価の補足

本書ではクラス予測や予測値の丸めを使う。ここでは、モデルの出力スコアと判定を分けて確認する。

$$
\widehat y_i(\tau)=\mathbf{1}[s_i\geq\tau]
$$

閾値$\tau$を高くすると、陽性となる入力の集合は小さくなる。
その結果、同じ評価集合ではFPとTPは増えず、FNは減らない。
適合率が必ず単調に上がるとは限らないので、実際の件数を見て判断する。

今回は、**検証上の誤検知率が1%以下**になる候補のうち、再現率が最大の閾値を選ぶ。

$$
\tau^*\in\underset{\tau}{\operatorname{arg\,max}}\ 
\operatorname{Recall}_{\mathrm{val}}(\tau)
\quad\text{subject to}\quad
\operatorname{FPR}_{\mathrm{val}}(\tau)\leq0.01
$$

同点なら高い閾値を選ぶ。陽性を1件も出さない閾値も候補に含む。
該当する閾値が得られても、未知データで誤検知率1%以下になる保証はない。
たとえば検証中の正常例が約550件なら、1件の増減でFPRは約0.18ポイント変わる。

閾値を決めたあとに訓練・検証を合わせて再学習すると、モデルのスコアも変わりうる。
そのため、この実装では再学習せず、選んだ閾値を同じモデルに適用する。
また、`predict_proba`の値を、そのまま実運用上の「悪性である確率」とは解釈しない。
確率の較正や実運用での攻撃混入率を検証していないためである。

In [ ]:
import json
from dataclasses import asdict

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

In [ ]:
def select_threshold(y, scores, max_fpr: float) -> float:
    y, scores = np.asarray(y), np.asarray(scores)
    if not 0 <= max_fpr <= 1:
        raise ValueError("max_fpr must lie in [0, 1]")
    if set(np.unique(y)) != {0, 1} or not np.isfinite(scores).all():
        raise ValueError("Threshold selection requires both classes and finite scores")
    fpr, tpr, thresholds = roc_curve(y, scores, drop_intermediate=False)
    eligible = np.flatnonzero(fpr <= max_fpr)
    # ROC thresholds descend; ties keep the more conservative operating point.
    best = eligible[np.argmax(tpr[eligible])]
    threshold = thresholds[best]
    return float(threshold if np.isfinite(threshold) else np.nextafter(scores.max(), np.inf))

In [ ]:
def binary_metrics(y, scores, threshold: float = 0.5) -> dict:
    prediction = np.asarray(scores) >= threshold
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y, prediction)),
        "precision": float(precision_score(y, prediction, zero_division=0)),
        "recall": float(recall_score(y, prediction, zero_division=0)),
        "f1": float(f1_score(y, prediction, zero_division=0)),
        "average_precision": float(average_precision_score(y, scores)),
        "roc_auc": float(roc_auc_score(y, scores)),
        "false_positive_rate": float(fp / (tn + fp)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [ ]:
def feature_importance(model, feature_names=None) -> pd.Series:
    classifier = model.named_steps["classifier"]
    if "tfidf" in model.named_steps:
        feature_names = model.named_steps["tfidf"].get_feature_names_out()
    if hasattr(classifier, "coef_"):
        values = np.abs(classifier.coef_[0])
    else:
        values = classifier.feature_importances_
    return pd.Series(values, index=feature_names, name="importance").sort_values(ascending=False)

In [ ]:
def save_report(output_dir, config, audits, studies, splits, metrics, hashes, revision):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    report = {
        "config": asdict(config),
        "positive_class": {"phishing": "UCI Result = -1", "spam": "enron1/spam"},
        "upstream_revision": revision,
        "dataset_sha256": hashes,
        "data_audit": {name: asdict(audit) for name, audit in audits.items()},
        "split_indices": {
            name: {k: v.tolist() for k, v in vars(s).items()} for name, s in splits.items()
        },
        "search": {
            name: {"best_cv_f1": study.best_value, "params": study.best_params}
            for name, study in studies.items()
        },
        "test_metrics": metrics,
        "versions": {
            name: version(name)
            for name in ("numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "optuna")
        },
    }
    (output_dir / "report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    for name, study in studies.items():
        study.trials_dataframe().to_csv(output_dir / f"{name}_trials.csv", index=False)
    return report

In [ ]:
tasks = {
    "logistic": (X_phishing, y_phishing, phishing_split),
    "tree": (X_phishing, y_phishing, phishing_split),
    "spam": (X_email, y_email, email_split),
}
thresholds, validation_metrics = {}, {}
for name, (X, y, split) in tasks.items():
    scores = models[name].predict_proba(X.iloc[split.validation])[:, 1]
    thresholds[name] = select_threshold(
        y.iloc[split.validation],
        scores,
        config.max_false_positive_rate,
    )
    validation_metrics[name] = binary_metrics(y.iloc[split.validation], scores, thresholds[name])
columns = ["threshold", "precision", "recall", "false_positive_rate"]
display(pd.DataFrame(validation_metrics).T[columns])

## 6. 混同行列から評価する — §2.2.1、pp.24–26

危険側を1、正常側を0とする。下の表は本書表2-1（p.25）と同じ向きで、
`confusion_matrix(..., labels=[0, 1])`の行・列にも対応する。

| 実際＼予測 | 正常：0 | 危険：1 |
|---|---|---|
| 正常：0 | TN：正常を通す | FP：正常を危険と誤検知 |
| 危険：1 | FN：危険を見逃す | TP：危険を検知 |

$$
\operatorname{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN},\qquad
\operatorname{Precision}=\frac{TP}{TP+FP},\qquad
\operatorname{Recall}=\frac{TP}{TP+FN}
$$

適合率は「警告したもののうち何割が本当に危険か」、再現率は「危険なものを何割拾えたか」である。
適合率と、正常例を誤検知した割合であるFPRは、分母が違う。

$$
\operatorname{F1}=\frac{2\operatorname{Precision}\operatorname{Recall}}
{\operatorname{Precision}+\operatorname{Recall}}
=\frac{2TP}{2TP+FP+FN},\qquad
\operatorname{FPR}=\frac{FP}{FP+TN}
$$

Accuracy・Precision・Recallは本書p.25の式に対応する。
F1は本書p.26で紹介される調和平均を式にしたもの、FPRはこの実習の閾値選択のための補足である。
陽性予測が0件などで分母が0になる適合率・再現率・F1は、実装の`zero_division=0`に合わせて0と表示する。

### 補足：攻撃が少ない環境では何件の警告になるか

1日10,000件、危険な入力が100件、正常が9,900件あると仮定する。
再現率90%、FPR 1%の検知器なら、件数は次のようになる。

| 実際＼予測 | 正常 | 危険 |
|---|---:|---:|
| 正常9,900件 | TN = 9,801 | FP = 99 |
| 危険100件 | FN = 10 | TP = 90 |

正解率は$(9801+90)/10000=98.91\%$と高いが、警告189件のうち正しいのは90件なので、適合率は約47.6%になる。
F1は$180/289\simeq0.623$である。
これは仮定した件数での計算例であり、今回の実測値ではない。

危険な入力の割合を$\pi$とすると、同じ再現率・FPRを仮定した適合率は次で計算できる。

$$
\operatorname{Precision}=
\frac{\pi\operatorname{Recall}}
{\pi\operatorname{Recall}+(1-\pi)\operatorname{FPR}}
$$

本書p.20の脚注も、ラベルの偏りと正解率の読み方を注意している。
この実習では多数派クラスだけを返す`DummyClassifier`も比較に入れ、
「常に同じ答えを返す場合に比べて何が良くなったか」を確認する。

### PR曲線、AP、ROC AUC

PR曲線は閾値を動かしたときの適合率と再現率の組を描く。
APはその順位全体をまとめる指標で、scikit-learnでは再現率の増分を重みとする適合率の和で計算する。
単純な台形面積と同一ではない。

$$
\operatorname{AP}=\sum_k(R_k-R_{k-1})P_k
$$

ROC AUCも閾値全体の順位性能を見るが、単一の運用閾値で何件誤検知するかまでは示さない。
次の出力では、閾値0.5と検証で決めた閾値を併記する。
まず採用モデルのFP/FNを読み、それから曲線でスコアの順位を確認する。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

test_metrics, test_scores = {}, {}
for name, (X, y, split) in tasks.items():
    scores = models[name].predict_proba(X.iloc[split.test])[:, 1]
    test_scores[name] = scores
    test_metrics[f"{name}/default"] = binary_metrics(y.iloc[split.test], scores)
    test_metrics[f"{name}/validation_threshold"] = binary_metrics(
        y.iloc[split.test],
        scores,
        thresholds[name],
    )
    baseline = DummyClassifier(strategy="most_frequent").fit(
        np.zeros((len(split.train), 1)),
        y.iloc[split.train],
    )
    baseline_scores = baseline.predict_proba(np.zeros((len(split.test), 1)))[:, 1]
    test_metrics[f"{name}/majority"] = binary_metrics(y.iloc[split.test], baseline_scores)
summary = pd.DataFrame(test_metrics).T
display(
    summary[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "average_precision",
            "false_positive_rate",
            "fp",
            "fn",
        ]
    ].round(4)
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, (name, (_, y, split)) in zip(axes, tasks.items(), strict=True):
    ConfusionMatrixDisplay.from_predictions(
        y.iloc[split.test],
        test_scores[name] >= thresholds[name],
        labels=[0, 1],
        display_labels=["benign", "malicious"],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(name)
fig.tight_layout()
output_dir = Path("artifacts") / "results"
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / "confusion_matrices.png", bbox_inches="tight")
plt.show()

fig, axis = plt.subplots(figsize=(6, 4))
for name, (_, y, split) in tasks.items():
    PrecisionRecallDisplay.from_predictions(
        y.iloc[split.test],
        test_scores[name],
        name=name,
        ax=axis,
    )
fig.tight_layout()
fig.savefig(output_dir / "precision_recall.png", bbox_inches="tight")
plt.show()

## 7. 特徴量重要度と`subject`の偏り — §2.3.2、pp.36–38

本書は、メール分類器で`subject`が強く寄与したことを受け、spamとhamでその出現数を調べている。
メールの内容を学習したつもりでも、引用の多さ、ヘッダーの残し方、送信者、時期など、
コーパスの作られ方に結び付いた特徴が手掛かりになることがある。

この実習では、特徴量重要度と、訓練メールの`subject`集計を並べて確認する。
集計は小文字化し、単語境界で`subject`を数える。書籍の部分文字列カウントと同じ条件ではない。

| 出力 | 読み方 |
|---|---|
| `messages` | クラスごとの訓練メール数 |
| `subject_tokens` | `subject`の延べ出現回数 |
| `messages_with_subject` | `subject`を1回以上含むメール数 |

延べ回数が多いだけでは、多くのメールに広く出るのか、少数の長い引用に集中しているのか区別できない。
クラスの件数自体も違うため、総数だけでなく1通あたりの回数や含有率も見る必要がある。

重要度の定義もモデルによって異なる。

| モデル | 表示する値 | 注意する点 |
|---|---|---|
| ロジスティック回帰 | 標準化後の係数の絶対値 | 危険側・正常側への向きは絶対値から読めない |
| 決定木 | 不純度減少に基づく重要度 | 相関する特徴量へ寄与が分散することがある |
| LightGBM | 分岐によるgainの合計 | 数値を他モデルの重要度と直接比較しない |

`subject`が上位に出るとは限らない。今回のデータ分割や探索で、別の単語が上位になっても誤動作ではない。
逆に`subject`が上位でも、その単語を足せば検知を回避できるとまでは分からない。
重要度は因果効果ではなく、実際に入力を変えた場合の予測も別途確かめる必要がある。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, (name, model) in zip(axes, models.items(), strict=True):
    importance = feature_importance(model, FEATURE_NAMES if name != "spam" else None)
    importance.to_csv(output_dir / f"{name}_importance.csv", header=True)
    importance.head(10).sort_values().plot.barh(ax=axis, title=name)
fig.tight_layout()
fig.savefig(output_dir / "feature_importance.png", bbox_inches="tight")
plt.show()

email_train = emails.iloc[email_split.train]
subject_audit = (
    email_train.assign(
        subject_count=email_train.text.str.lower().str.count(r"\bsubject\b"),
    )
    .groupby("label")
    .agg(
        messages=("text", "size"),
        subject_tokens=("subject_count", "sum"),
        messages_with_subject=("subject_count", lambda values: int(values.gt(0).sum())),
    )
)
display(subject_audit.rename(index={0: "ham", 1: "spam"}))

## 8. 推論と結果の保存 — §2.4、p.38

最後に、短い英語テキストをPipelineへ渡す。
学習時に決めた語彙でTF-IDFへ変換し、LightGBMのスコアを求め、検証で選んだ閾値を適用する。
未知の単語は新たに語彙へ追加せず、変換時には無視される。

この手入力例には正解ラベルを付けていないので、表示される結果から正解率は計算できない。
例文を2つ分類できたことと、実メールで性能を確認したことも別になる。
古い英語コーパスでの結果を、日本語メールや現在のフィッシングへそのまま移して考えないようにしたい。

`report.json`には、乱数、分割、重複除去、バージョン、探索結果、指標を記録する。
同じ正解率でも、分割やラベルの定義が違えば同じ実験にはならない。
ゼミで結果を比べるときは、数字と一緒にこれらの条件を残す。
ZIPにはメール本文や学習済みpickleを含めない。

In [ ]:
import shutil

messages = pd.Series(
    [
        "Please review the attached project agenda before tomorrow's meeting.",
        "Congratulations! Claim your free prize now. Limited offer, click to collect.",
    ]
)
scores = models["spam"].predict_proba(messages)[:, 1]
display(
    pd.DataFrame(
        {
            "text": messages,
            "spam_score": scores,
            "predicted_spam": scores >= thresholds["spam"],
        }
    )
)

report = save_report(
    output_dir,
    config,
    {"phishing": phishing_audit, "enron1": email_audit},
    studies,
    {"phishing": phishing_split, "enron1": email_split},
    test_metrics,
    DATA_HASHES,
    UPSTREAM_REVISION,
)
summary.to_csv(output_dir / "test_metrics.csv")
subject_audit.to_csv(output_dir / "subject_audit.csv")
(output_dir / "selected_models.json").write_text(
    json.dumps({"phishing": selected_phishing, "spam": "spam", "thresholds": thresholds}, indent=2),
    encoding="utf-8",
)
archive_path = shutil.make_archive("security-ml-results", "zip", output_dir)
print(f"Results: {archive_path}")

必要なら次のセルでZIPを手元に保存する。学習済みpickleやメール本文は含めず、
指標、分割インデックス、探索履歴、重要度、図、実行バージョンを記録する。

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(archive_path)

## ゼミで結果を読むときの確認事項

**1．決定木とロジスティック回帰の差は、何に由来しそうか。**

同じ分割でのCV結果を使って、線形境界と条件分岐の違いから仮説を立てる。
最終テストを見ながら何度も探索範囲を変えると、そのテストへ実験手順が適合していく。
追加実験をするなら訓練内で比較し、最終テストは新たに確保する。

**2．誤検知率を下げたとき、見逃しは何件増えたか。**

`default`と`validation_threshold`の行を比べ、FPとFNを件数で読む。
通常メールの誤隔離と攻撃メールの見逃しに、それぞれどの程度の運用コストを置くかによって、採用する閾値は変わる。
F1だけでその判断を代用できるとは限らない。

**3．本文の内容より、データの由来を学習していないか。**

重要語を、件名、引用、組織名、時期を表す語に分けて眺める。
それらを除く実験を行うなら、まず訓練内で検証する。
時期や送信者をまたぐ評価ができると、ランダム分割とは異なる弱点が見える可能性がある。

### §2.5の演習へ進む場合

本書p.39はLing-Spamで同じ分類手順を組む演習である。
このノートブックにはそのダウンロードや学習までは追加していない。
取り組む場合は、ラベル付けと重複の扱いを確認し、分割してからTF-IDFを学習する。
また、次の2つは異なる実験なので、結果を分けて報告する。

| 実験 | 調べられること |
|---|---|
| Ling-Spam内で訓練・評価する | そのコーパスの同じ分布に近いデータへの性能 |
| Enronで訓練したモデルを固定してLing-Spamで評価する | コーパスが変わった場合の性能 |

後者で評価用コーパスの語彙を学習し直すと、固定したモデルの転用評価ではなくなる。
前処理もモデルに含めて固定する、という点は第1章の標準化から一貫している。

## 出典

- [書籍サンプル・正誤情報](https://github.com/oreilly-japan/ml-security-jp)
- [UCI Phishing Websites](https://doi.org/10.24432/C51W2X) — R. Mohammad, F. Thabtah, L. McCluskey
- [Enron-Spam](https://www2.aueb.gr/users/ion/data/enron-spam/)
  — V. Metsis, I. Androutsopoulos, G. Paliouras (2006)

§2.5のLing-Spam演習は本ノートブックの対象外。